# Naqshkaar 3D worker on Google Colab

Runs a generation worker on a Colab GPU and connects it to your Catalogue Studio.
TripoSR fits a free T4. TRELLIS needs an A100 or L4 (Colab Pro).

**Before you start**
1. Run the studio on your machine (`docker compose up`).
2. Expose its API so Colab can reach it, for example with Cloudflare's quick tunnel:
   `cloudflared tunnel --url http://localhost:8000` and copy the `https://….trycloudflare.com` URL.
3. In the studio, go to **Settings → API keys** and create a key with the **worker** scope.


In [ ]:
!nvidia-smi

## 1. Get the worker code
Upload `naqshkaar-catalogue-studio.zip` with the file browser on the left, or clone your fork.

In [ ]:
import os, zipfile
if os.path.exists('/content/naqshkaar-catalogue-studio.zip'):
    zipfile.ZipFile('/content/naqshkaar-catalogue-studio.zip').extractall('/content')
# or: !git clone https://github.com/<your-user>/Naqshkaar-FYP.git /content/repo
%cd /content/naqshkaar-catalogue-studio/generator
!pip -q install -e .

## 2. Install an engine
Run **one** of the cells below.

In [ ]:
# TripoSR (free T4)
!git clone -q https://github.com/VAST-AI-Research/TripoSR.git /content/TripoSR
!pip -q install -r /content/TripoSR/requirements.txt rembg onnxruntime-gpu
os.environ['TRIPOSR_REPO'] = '/content/TripoSR'
ENGINE = 'triposr'

In [ ]:
# Hunyuan3D-2mini shape only (T4 works with texture=false)
!git clone -q https://github.com/Tencent-Hunyuan/Hunyuan3D-2.git /content/Hunyuan3D-2
!pip -q install -r /content/Hunyuan3D-2/requirements.txt && pip -q install -e /content/Hunyuan3D-2
ENGINE = 'hunyuan3d2'   # in the studio, set params: {"model": "tencent/Hunyuan3D-2mini", "subfolder": "hunyuan3d-dit-v2-mini", "texture": false}

## 3. Connect and run
Paste your tunnel URL and worker key. Leave this cell running; jobs appear in the studio's **Jobs** page.

In [ ]:
os.environ['STUDIO_URL'] = 'https://YOUR-TUNNEL.trycloudflare.com'
os.environ['STUDIO_API_KEY'] = 'nqk_...'
os.environ['WORKER_ID'] = 'colab-' + ENGINE
os.environ['ENGINES'] = ENGINE
os.environ['GPU_LOCK_FILE'] = ''
!python -m naqshkaar_generator --list-engines
!python -m naqshkaar_generator